# Example Notebook for a basic vignette for `pytorch-forecasting v2` Model Training and Inference

<div class="alert alert-block alert-info">
:warning: The "Data Pipeline" showcased here is part of an experimental rework of the `pytorch-forecasting` data layer, planned for release in v2.0.0. The API is currently unstable and subject to change without prior notice. This notebook serves as a basic demonstration of the intended workflow and is not recommended for use in production environments. Feedback and suggestions are highly encouraged — please share them in <a href="https://github.com/sktime/pytorch-forecasting/issues/1736">issue 1736</a>.
</div>


In this notebook, we demonstrate how to train and evaluate the **Temporal Fusion Transformer (TFT)** using the `pytorch-forecasting v2` architecture.
We can do this in 2 ways:
1. **High-Level Forecaster API:**

    This approach provides a clean, `scikit-learn`-like `fit` → `predict` workflow. It manages data transformation, metadata inspection, and Lightning `Trainer` execution under a unified interface while allowing full injection of custom `Trainer` and `DataModule` configurations.

2. **Low-Level 3-Stage Pipeline:**
This involves explicitly constructing:
    * a `TimeSeries` object

    * a `DataModule`

    * the model (e.g., `TFT`)
    
    This workflow is ideal if you need custom setups such as custom trainers, callbacks, or advanced data preprocessing.
    It requires a deeper understanding of how the three layers (TimeSeries, DataModule, and the model) interact, but offers maximum flexibility.

## Create Synthetic data
We generate a synthetic dataset using `load_toydata` that creates a `pandas` DataFrame with just numerical values as for now **the pipeline assumes the data to be numerical only**.

In [1]:
from pytorch_forecasting.data.examples import load_toydata

In [2]:
num_series = 100  # Number of individual time series to generate
seq_length = 50  # Length of each time series
data_df = load_toydata(num_series, seq_length)
data_df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,-0.068801,-0.006352,0,1.000000,0.587279,0
1,0,1,-0.006352,0.524402,0,0.995004,0.587279,0
2,0,2,0.524402,0.644409,0,0.980067,0.587279,0
3,0,3,0.644409,0.522688,0,0.955336,0.587279,0
4,0,4,0.522688,0.937309,0,0.921061,0.587279,0


## High-level API


### Steps
* Create the `TimeSeries` dataset object
* Configure components (DataModule and Trainer) and instantiate `TFTForecaster`
* Perform `forecaster.fit` to train the model
* Perform `forecaster.predict` to generate forecasts as a `TimeSeries` object

###  Create Dataset object

`TimeSeries` holds the data frame and its schema.

---

`TimeSeries` dataset's Key arguments:
- `data`: DataFrame with sequence data.
- `time`: integer typed column denoting the time index within `data`.
- `target`:  Column(s) in `data` denoting the forecasting target.
- `group`: List of column names identifying a time series instance within `data`.
- `num`: List of numerical features.
- `cat`: List of categorical features.
- `known`: Features known in future
- `unknown`: Features not known in the future
- `static`: List of variables that do not change over time,



In [3]:
from pytorch_forecasting.data.timeseries import TimeSeries

In [4]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


#### `TimeSeries` is the data container

`TimeSeries` holds the data for both input and output. `fit` and `predict` take a `TimeSeries`, and `predict` returns one.

* `to_pandas()`: returns the data as a data frame
* `get_metadata()`: returns the schema
* `dataset[i]`: returns series `i` as a `dict`

In [5]:
# `TimeSeries` is the data container of ptf
# a data frame plus its schema
print("is_prediction:", dataset.get_metadata().is_prediction)
print("Columns:", dataset.get_metadata()["cols"])

dataset.to_pandas().head()

is_prediction: False
Columns: {'y': ['y'], 'x': ['x', 'category', 'future_known_feature', 'static_feature', 'static_feature_cat'], 'st': ['static_feature', 'static_feature_cat']}


,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,-0.068801,-0.006352,0,1.000000,0.587279,0
1,0,1,-0.006352,0.524402,0,0.995004,0.587279,0
2,0,2,0.524402,0.644409,0,0.980067,0.587279,0
3,0,3,0.644409,0.522688,0,0.955336,0.587279,0
4,0,4,0.522688,0.937309,0,0.921061,0.587279,0


### Configure Components & Model Parameters


In [6]:
from lightning.pytorch import Trainer
from sklearn.preprocessing import StandardScaler

from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.encoders import (
    NaNLabelEncoder,
    TorchNormalizer,
)
from pytorch_forecasting.metrics import MAE, SMAPE

Here we configure the `EncoderDecoderTimeSeriesDataModule`.

`EncoderDecoderTimeSeriesDataModule` key arguments:
- `time_series`: `TimeSeries` dataset instance (optional during initial forecaster setup; bound at fit time).
- `max_encoder_length`: Maximum length of the encoder input sequence.
- `max_prediction_length`: Maximum length of the decoder output sequence.
- `batch_size`: Batch size for DataLoader.
- `categorical_encoders`: Dictionary of categorical encoders.
- `scalers`: Dictionary of feature scalers.
- `target_normalizer`: Normalizer for the target variable.

In [7]:
# Configure the datamodule with preprocessing scalers and sequence lengths
datamodule = EncoderDecoderTimeSeriesDataModule(
    max_encoder_length=30,
    max_prediction_length=3,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

/tmp/ipykernel_23623/2781251925.py:2: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  datamodule = EncoderDecoderTimeSeriesDataModule(


We configure a PyTorch Lightning `Trainer` to handle hardware acceleration, logging, and epoch training loops.

In [8]:
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [9]:
from pytorch_forecasting.models.temporal_fusion_transformer import TFTForecaster

### Create the `TFTForecaster` object

The `TFTForecaster` acts as the high-level forecaster for the Temporal Fusion Transformer. It combines model parameters (such as `hidden_size`, `num_layers`, and `loss`) with our configured `Trainer` and `DataModule`, providing a unified `fit` and `predict` interface.

In [10]:
forecaster = TFTForecaster(
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    trainer=trainer,
    datamodule=datamodule,
)

In [11]:
# Train the forecaster on the dataset
forecaster.fit(dataset)

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:221: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(time_series=data, **self._init_kwargs)
/home/aryan/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecas

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

... (2 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


Artifacts saved in: /home/aryan/pytorch-forecasting/docs/source/tutorials/checkpoints


PosixPath('/home/aryan/pytorch-forecasting/docs/source/tutorials/checkpoints/best-epoch=4-step=190.ckpt')

#### Pass the trainer to `fit`

* You can pass the trainer to `__init__` or to `fit`.
* If you pass both, `fit` uses the one passed to `fit`.
* `forecaster.trainer_` is the trainer used in the last `fit`.
* The cell below trains a second forecaster, so `forecaster` stays as it is.

In [12]:
fit_trainer = Trainer(max_epochs=1, accelerator="auto", devices=1)

other_forecaster = TFTForecaster(datamodule=datamodule)
other_forecaster.fit(dataset, trainer=fit_trainer)

print("Used the trainer from fit:", other_forecaster.trainer_ is fit_trainer)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:221: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(time_series=data, **self._init_kwargs)
/home/aryan/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an expe

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:321: The numbe

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


Artifacts saved in: /home/aryan/pytorch-forecasting/docs/source/tutorials/checkpoints
Used the trainer from fit: True


#### Prediction Output

`predict()` returns a `TimeSeries`, the same type as the input. Its table has one row per forecast step and these columns.

* `series_id`, `time_idx`: group and time columns of the input data
* `y`: the forecast
* `time_idx_origin`: last time step the model saw before this forecast
* `y_actual`: the actual value, if `"y"` is in `return_info`

Other `return_info` outputs, such as `"x"` and `"index"`, are in `preds.info`.

In [13]:
preds = forecaster.predict(dataset, return_info=["x", "y"])

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:221: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(time_series=data, **self._init_kwargs)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelC

Predicting: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


In [14]:
print("Prediction TimeSeries object:", preds)
print("\nForecasts Table (first 5 rows):")
print(preds.data.head())

Prediction TimeSeries object: <pytorch_forecasting.data.timeseries._timeseries_v2.TimeSeries object at 0x740fbc12c890>

Forecasts Table (first 5 rows):
   series_id  time_idx         y  time_idx_origin  y_actual
0          0        30  0.032210               29  0.000829
1          0        31  0.223469               29 -0.077484
2          0        32  0.419073               29  0.350177
3          0        31  0.095694               30 -0.077484
4          0        32  0.289567               30  0.350177


#### Predict on new data

* `predict` takes a new `TimeSeries` with the same schema as the training data.
* A different schema raises a `ValueError`.
* `predict` calls `with_data` on the fitted datamodule, so you do not call it yourself.
* `predict` uses the transforms fitted in `fit` and does not refit them.

In [16]:
# new data with the same schema, scaled with the transforms fitted in `fit`
new_dataset = TimeSeries(
    data=load_toydata(20, seq_length),
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

new_preds = forecaster.predict(new_dataset)
new_preds.to_pandas().head()

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:221: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(tim

Predicting: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


,series_id,time_idx,y,time_idx_origin
0,0,30,0.021175,29
1,0,31,0.235371,29
2,0,32,0.465765,29
3,0,31,0.109364,30
4,0,32,0.313771,30


## 3-stage pipeline




### Steps

1. Create `TimeSeries` Dataset object
2. Create DataModule object
3. Initialize, Train & Run Inference with the Model




####  Create Dataset & DataModule

- `TimeSeries` returns the raw data in terms of tensors .
- `DataModule` wraps the dataset, handles splits, preprocessing, batching, and exposes `metadata` for the model initialisation.



#### Initialize the Model

We initialize the TFT model using the `metadata` provided by the `DataModule`. This metadata includes all required dimensional info for the encoder, decoder, and static inputs.



#### Train the Model

We use a `Trainer` from PyTorch Lightning to train the model

#### Run Inference

After training, we can make predictions using the trained model


### 1. Create the dataset
We create a `TimeSeries` dataset instance that returns the raw data in terms of tensors, then this "raw data" is sent to the `data_module`that will internally handle the dataloaders and preprocessing

`TimeSeries` dataset's Key arguments:
- `data`: DataFrame with sequence data.
- `time`: integer typed column denoting the time index within `data`.
- `target`:  Column(s) in `data` denoting the forecasting target.
- `group`: List of column names identifying a time series instance within `data`.
- `num`: List of numerical features.
- `cat`: List of categorical features.
- `known`: Features known in future
- `unknown`: Features not known in the future
- `static`: List of variables that do not change over time,

In [17]:
from pytorch_forecasting.data.timeseries import TimeSeries

In [18]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


### 2. Create datamodule

`EncoderDecoderTimeSeriesDataModule` key arguments:
- `time_series`: `TimeSeries` dataset instance
- `max_encoder_length`: Maximum length of the encoder input sequence.
- `max_prediction_length`: Maximum length of the decoder output sequence.
- `batch_size`: Batch size for DataLoader.
- `categorical_encoders`: Dictionary of categorical encoders.
- `scalers`: Dictionary of feature scalers.
- `target_normalizer`: Normalizer for the target variable.

In [19]:
from sklearn.preprocessing import StandardScaler
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.encoders import (
    NaNLabelEncoder,
    TorchNormalizer,
)

In [20]:
# create the `data_module` that handles the dataloaders and preprocessing
data_module = EncoderDecoderTimeSeriesDataModule(
    time_series=dataset,
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

/tmp/ipykernel_23623/1675030273.py:2: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  data_module = EncoderDecoderTimeSeriesDataModule(


### 3. Initialise and train the model

To initialise the model you now don't have to pass arguments like `encoder_cont`, `decoder_cont` etc as they are calculated internally using the `metadata` property [[source]](https://github.com/sktime/pytorch-forecasting/blob/4a34931e499c2b59de3939fcffcaabd75204b045/pytorch_forecasting/data/data_module.py#L264-L292) of `EncoderDecoderTimeSeriesDataModule`. But you still have to pass other params like `loss`, `optimizer` etc


```python
model = TFT(
    loss=nn.MSELoss(),
    logging_metrics=[MAE(), SMAPE()],
    metadata=data_module.metadata,  # <-- crucial for model setup
    ...
)
```

The `metadata` includes:
- `max_encoder_length`, `max_prediction_length`
- number of continuous/categorical variables in encoder/decoder
- number of static features

These are used to configure internal layers like `encoder_cont`, `decoder_cat`, etc.


In [21]:
from pytorch_forecasting.metrics import MAE, SMAPE
from pytorch_forecasting.models.temporal_fusion_transformer._tft_v2 import TFT

In [22]:
# Initialise the Model
model = TFT(
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
    metadata=data_module.metadata,  # pass the metadata from the datamodule to the model
    # to initialise important params like `encoder_cont` etc
)

/home/aryan/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


We use a `Trainer` from PyTorch Lightning to train the model:

```python
trainer = Trainer(max_epochs=5, ...)
trainer.fit(model, data_module)
```

The `Trainer`:
- Pulls data from `data_module`
- Handles device placement
- Logs training progress and metrics


In [23]:
from lightning.pytorch import Trainer

In [24]:
# Train the model
print("\nTraining model...")
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

trainer.fit(model, data_module)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.



Training model...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------------------
0 | loss                  | MAE                | 0      | train | 0    
1 | logging_metrics       | ModuleList         | 0      | train | 0    
2 | encoder_var_selection | Sequential         | 709    | train | 0    
3 | decoder_var_selection | Sequential         | 193    | train | 0    
4 | static_context_linear | Linear             | 192    | train | 0    
5 | lstm_encoder          | LSTM               | 51.5 K | train | 0    
6 | lstm_decoder          | LSTM               | 50.4 K | train | 0    
7 | self_attention        | MultiheadAttention | 16.6 K | train | 0    
8 | pre_output            | Linear             | 4.2 K  | train | 0    
9 | output_layer          | Linear             | 65     | train | 0    
-----------------------------------------------------------------------------
123 K    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

... (2 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.



#### Output
Output of TFT model is a `dict` with key `prediction`:

- `y_pred["prediction"]`: Tensor of shape `(batch_size, prediction_length, output_size)`


In [25]:
data_module.setup(stage="test")
test_dataloader = data_module.test_dataloader()

In [26]:
preds = model.predict(test_dataloader, return_info=["index", "x", "y"])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider i

Predicting: |          | 0/? [00:00<?, ?it/s]

In [27]:
print("First Predicted Value:")
print("Index:", preds["index"][0].item())
print("Prediction:", preds["prediction"][0].item())
print("Actual:", preds["y"][0].item())

First Predicted Value:
Index: -0.09024101495742798
Prediction: -0.37452489137649536
Actual: -0.7080494165420532


#### Predict on new data

* `with_data` returns a new datamodule with the new data. It keeps the parameters and fitted transforms of `data_module`.
* `trainer.predict` returns one `dict` per batch, in the normalized scale.
* `inverse_transform_target` returns the values in the original scale.
* `TimeSeries.from_tensors` puts the tensors in a `TimeSeries`, the same type as the input. Its columns are `_series`, `_time_idx` and the target.

In [28]:
import torch

# `new_dataset` is the `TimeSeries` created in the high-level section
pred_dm = data_module.with_data(new_dataset)

# one dict per batch
batches = trainer.predict(model, datamodule=pred_dm)
y = torch.cat([batch["prediction"] for batch in batches])

# back to the original target scale
windows = pred_dm.predict_windows
y = pred_dm.inverse_transform_target(y, windows)

# one row per window here, as `max_prediction_length=1`
preds_ts = TimeSeries.from_tensors(
    {
        "y": y.reshape(-1, 1),
        "t": torch.tensor([start + enc_len for _, start, enc_len, _ in windows]),
    },
    metadata=new_dataset.get_metadata(),
    groups=[series for series, *_ in windows],
)
preds_ts.to_pandas().head()

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:221: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(time_series=data, **self._init_kwargs)
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many wor

Predicting: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


,_series,_time_idx,y
0,0,30,0.034066
1,0,31,0.216841
2,0,32,0.452285
3,0,33,0.615383
4,0,34,0.751750
